# Knee MRI — preprocessing & model pipeline for IMAGES

## 0. Reality check — five things we had wrong

Correcting these now is cheaper than correcting them.

| # | What we assumed | What is actually true |
|---|---|---|
| 1 | "Each **series** gives 3 arrays: X / Y / Z, ~40 images each" | A series is **one** acquisition in **one** plane — that is why `train_series.csv` has an `Anatomical_Plane` column. A study averages **~5.5 series**. Our unit of work is the **study**, reduced to **4 fixed slots** (§2). Reslicing one plane into the others fabricates resolution: through-plane is 3–4 mm vs 0.3–0.6 mm in-plane, and the study already contains a real acquisition in each plane. |
| 2 | "One specialised **model** per label" | Right instinct, wrong unit. 12 fine-tuned backbones × ~1,300 test studies will not fit the **9-hour** GPU limit. One shared encoder + **12 specialised heads** gives each label its own attention, threshold and colour map at 1/12 of the cost. See §5. |
| 3 | "Train on images only, ignore reports, don't trust the 58" | With the 58 excluded and reports excluded there are **zero** labels. Resolution: the reports become a **label file**, not a module — see §0.1. |
| 4 | "The model returns a knee with a colour spot on the injury" | There are **no masks and no boxes** in this dataset, only study-level 0/1. The colour must come from **weakly-supervised** localisation (CAM / MIL attention). Cheaper to build, but coarse and not always right — §7 gates it. |
| 5 | "Download the data" | It is **569 GB**, and our competition-download endpoint is 429-locked. We use a **17.4 GB public preprocessed mirror** instead, and Kaggle GPU for the heavy pass. §2. |

### Where supervision actually comes from

- **58 of 4,407** studies have all 12 labels. The other **4,349** have a `Report` and nothing else.
- The 58 are **radiologist-adjudicated** — two MSK radiologists plus an adjudicator, the same
  process that produced the hidden test ground truth. They are the *most* reliable labels we have.
- But every one of the 58 has ≥1 positive finding, so the set is **selection-biased**, and 58
  across folds is ~12 per fold where the standard error of an AUC is ≈0.15.
  → **Never train on them. Always audit against them.**
- `test.csv` has **no Report column**. Any text branch is a *training-time teacher only*.<br> ⚠️ *Need word Team ?*
  Treating report text as a test feature is the classic fatal error in this competition.
  Module `M3` enforces this with an assertion (§4).

> ### Two things before the plan
>
> **`⚠️ *Need word Team ?*`** <br>marks every point where words from the radiology reports reach the
> model. Each one is a hand-off with the reports team: a place where a better extractor
> changes what we train on, and a place where text must *not* leak into inference.
> `test.csv` has exactly one column, `StudyInstanceUID` — verified in
> `img_preprocess_all_models.ipynb`.
>
> **The images are the critical path** <br>Nothing renders and nothing trains
> on pixels until M2 turns DICOM into canonical volumes. The reports work runs in parallel
> and can start from public label CSVs on day 1; the preprocessing cannot be parallelised
> away, and every stage in §6 waits on it.

---
## 1. Architecture at a glance

```
  ┌─────────────── KAGGLE ONLY — 569 GB already mounted at /kaggle/input ───────────────┐
  │                                                                                     │
  │  train_series.csv ─► M1 assign_slots ─► slots.csv ─► M2 dicom→volume ─► volumes.npy │
  │   24,371 rows          pandas, 30 s      4407 × 4      CPU kernel        uint8       │
  │   plane + fluid        NO dicom reads                  resumable      (4,24,224,224) │
  │                                                                            │        │
  │                                                    M4 encode  ◄────────────┘        │
  │                                                 GPU, DINOv2-S frozen                │
  └────────────────────────────────────────────────────────┬────────────────────────────┘
                                                           ▼
   train.csv ─► M3 label factory ─► soft_targets.csv          ⚠️ *Need word Team ?*   features.npy  [N,4,K,D]  ~600 MB
    (Report)    merge public LLM     Y (soft) + W       published as a Kaggle dataset,
                label CSVs           (W=0 ⇒ masked)     then pulled to the laptops
                                          │                    │
                                          └─────────┬──────────┘
                                                    ▼
                              M5 heads   —   LAPTOP, ~20 s per experiment
                              StandardScaler → PCA(512) → 12 × weighted Ridge
                                                    │
                           ┌────────────────────────┴────────────────────────┐
                           ▼                                                 ▼
              submit kernel (GPU, internet OFF)                  M5 cam ─► M6 render packs
              decodes the HIDDEN test from DICOM                 exact linear decomposition
                           │                                                 │
                           ▼                                                 ▼
                    submission.csv                              app/  2D tri-planar + 3D
                   (Kaggle: 12 AUCs)                            + 12 confidence bars
```

---
## 2. Data

| Artifact | Path | Shape / dtype | Size |
|---|---|---|---|
| competition CSVs | `data/*.csv` | verbatim | ~25 MB |
| LLM report labels ⚠️ *Need word Team ?* | `data/labels_llm_*.csv` | 4,407 × 13 | <1 MB |
| series manifest | `data/manifest/manifest_series.parquet` | 24,371 rows | ~8 MB |
| **canonical volume** | `data/processed/<profile>/<study>/<series>.npy` | `uint8 (24,224,224)` | 1.2 MB |
| slot assignment | `data/manifest/slots.csv` | 4,407 × 5 | 600 KB |
| study tensor | *(in memory, built from 4 slots)* | `uint8 (4,24,224,224)` | — |
| **soft targets** ⚠️ *Need word Team ?* | `data/derived/soft_targets.csv` | 4,407 × 40 | ~2 MB |
| label audit | `data/derived/label_audit.csv` | 12 × 8 | 2 KB |
| folds | `data/derived/study_folds.csv` | 4,407 × 4 | 400 KB |
| **cached features** | `data/features/features_<tag>.npy` | `float16 [N,4,K,D]` | ~600 MB |
| feature mask | `data/features/mask_<tag>.npy` | `bool [N,4,K]` | 265 KB |
| heat maps | `data/_heat/<study>.npz` | `uint8 (12,S,gh,gw)` | ~14 KB |
| **render pack** | `data/render_packs/<study>.npz` | `gray_iso uint8 128³`<br>`heat_iso uint8 (12,64³)` | ~5 MB |
| submission | `/kaggle/working/submission.csv` | 1,300 × 13 | ~150 KB |

**Frozen constants**

```python
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
           "Medial OA", "Lateral OA", "PF OA", "Effusion",
           "Synovitis", "Baker's", "Contusion", "Fracture"]   # submission order. NEVER sort.

# A study averages ~5.5 series. We take 4 fixed SLOTS,
# because fluid-sensitive vs structural matters as much as the plane:
#   Contusion / Fracture / Effusion are invisible outside a fluid-sensitive sequence.

SLOTS = [("SAG_FLUID",  "Sagittal", 1),   # 94.2% of studies have one
         ("SAG_STRUCT", "Sagittal", 0),   # 96.8%
         ("COR_FLUID",  "Coronal",  1),   # 96.4%
         ("AX_FLUID",   "Axial",    1)]   # ~100%
         
# AX_STRUCT is DROPPED on COST, not on masking. A 5th slot is +25% encoder time on every one
#   of the ~1,300 test studies (124,800 -> 156,000 forwards) to serve 19.4% of them. Missing
#   slots cost nothing to learn: present_mask zeroes them and the head masks them out.
# COR_STRUCT (77.3%) is dropped too, and that one is arguable — it sits with the kept slots,
#   not with AX_STRUCT, and coronal T1/PD is where the meniscal body and compartment cartilage
#   are read: 4 of 12 labels. Revisit if the meniscus or OA heads underperform.
# Fat_Suppression: the COLUMN is dropped, the AXIS is not. Identical to Fluid_Sensitive on all
#   24,371 train rows and all 15 test rows, but the two are physically independent and the
#   delivered column collapsed them. Recover it from the header: TR/TE -> T1/T2/PD weighting,
#   fat-sat from SeriesDescription/ScanOptions.
#   BLOCKING for Contusion — §5 says the oedema is invisible outside STIR/FS, so that head
#   cannot select its series without this axis.
#   Re-assert the equality on the test manifest, warn not fail: 15 rows prove nothing.

DEPTH, SIZE, CROP_MM = 24, 224, 130.0   # 0.580 mm/px. 224 = 16x14 -> DINOv2 ViT-S/14 compatible
WINDOW_PCT = (0.5, 99.5)                # per SERIES, on the cropped foreground
CACHE_TAG  = "v1_d24_s224_c130_4slot"   # in every filename; a mismatched tag REFUSES to load
```

**Serialisation rule:** `.npy` / `.npz` / `.csv` only.

---
## 3. Module pseudocode


### — Data access & materialisation

In [ ]:
# src/data/kaggle_pull.py

def preflight():
    # print versions, CACHE_TAG, where() -> kaggle | colab | local
    ...

def census(train_df, series_df):
    # slot fill rate per (plane, fluid); drop any slot below 0.90
    # is Fluid_Sensitive == Fat_Suppression? count fully-labelled studies (print, don't assert)
    # count duplicate Reports -> they leak across folds
    ...

TIERS = {
    "T0_meta":  "competition + LLM label CSVs   ~25 MB   laptop",
    "T1_dicom": "8 studies of raw DICOM         ~1 GB    laptop",
    "T2_gold":  "the 58 adjudicated studies     ~1.4 GB  laptop",
    "T3_full":  "preprocessed .npz mirror       17 GB    Kaggle/Colab",
}

def authenticate():
    # .env: KAGGLE_USERNAME + KAGGLE_API_TOKEN, as in kaggleFetch.ipynb
    ...

def pull_tier(tier, dest="data/"):
    # got data downloaded, as error 429 anytime api fetch failed
    ...

def build_manifest(split="train"):
    # one row per series
    # -> manifest_series.parquet
    ...

def verify_tier(tier):
    # sha256 every file against DATASETS.lock.json so all four of us have identical bytes
    ...

### M2 — DICOM → canonical volume  *(the core preprocessing)*
**goal** every study becomes one comparable `uint8 (3,24,224,224)` tensor
**in** a series directory · **out** `data/processed/<profile>/<study>/<series>.npy`

Five traps live in this module. Each line below exists because of one.

In [23]:
# src/preprocess/dicom_to_volume.py

def read_series(series_dir):
    # order slices by ImagePositionPatient on the normal, never by filename
    # pylibjpeg required: JPEG-Lossless and JPEG-2000 both occur
    # -> RawSeries(vol int16 (S,H,W), spacing, dirs, meta)
    ...

def normalize_intensity(vol):
    # one 1-99.5 percentile window per SERIES, pooled over slices
    # -> uint8
    ...

def crop_to_physical_extent(vol, spacing, mm=130):
    # PixelSpacing varies 3.4x -> crop to constant mm, then resize
    ...

def resolve_study_laterality(study_id, series_df):
    # decide the side once per STUDY, else planes disagree on which side is medial
    # -> (side in {L,R,None}, source, confident)
    ...

def fix_laterality(vol, plane, side, confident):
    # coronal/axial -> flip last axis; sagittal -> reverse stack order
    # not confident -> do nothing, a wrong flip is worse than none
    ...

def preprocess_series(series_dir, profile, side, confident, csv_plane=None):
    # read -> normalize -> crop -> resize -> fix_laterality, cv2.INTER_AREA
    # meta carries mm_per_px and slice_spacing_mm; they vary 3.4x, never constants
    # -> (uint8 (24,224,224) | None, meta)
    ...

def assign_slots(series_df):
    # pick 4 series per study from train_series.csv alone, no DICOM
    # -> slots.csv: study_uid, SAG_FLUID, SAG_STRUCT, COR_FLUID, AX_FLUID
    ...

def build_study_tensor(study_id, slots_row, profile):
    # missing slot -> zeros + present_mask 0, never impute
    # -> (uint8 (4,24,224,224), present_mask (4,), meta)
    ...

PROFILES = {
    "default": dict(mm=130, size=224, slices=24),   # full corpus
    "hires":   dict(mm=130, size=384, slices=32),   # gold-58, local dev
    "context": dict(mm=200, size=224, slices=24),   # STRETCH: full FOV for Baker's
}

### M3 — Supervision: reports → soft targets  <br>⚠️ *Need word Team ?*
**goal** turn 4,349 report-only studies into trainable soft labels **with a trust weight**
**in** `train.csv`, public LLM label CSVs · **out** `soft_targets.csv`, `label_audit.csv`

This is the highest-leverage module in the project and the one most likely to be built wrong.
**Every function below is a reports-team hand-off.** <br>⚠️ *Need word Team ?*

In [24]:
# src/labels/report_labels.py

# ⚠️ *Need word Team ?*  <- which extractor produces the targets
def load_public_llm_labels():
    # published LLM labels cover all 4,407 studies; no NLP pipeline of our own
    # use the single best source, not a blend
    # keep pilkwang for __conf/__verdict as this is the only published trust signal
    ...

# ⚠️ *Need word Team ?*  <- acceptance test for any extractor: beat 0.893 macro-AUC
def audit_against_gold(soft, gold_58):
    # per label: agreement, false_pos (negation missed), false_neg (never dictated)
    # -> label_audit.csv
    ...

# ⚠️ *Need word Team ?*  <- silence -> W=0; the extractor must separate NO from UNK
def trust_weights(audit):
    # silence: Medial Meniscus 5.7 .. Fracture 56.4, Synovitis 84.2
    # a silent cell gets W = 0, not a prior — it is a MISSING label, so masked BCE
    # -> W float32 (N,12), Y float32 (N,12)
    ...

# ⚠️ *Need word Team ?*  <- report_md5 de-duplication needs the raw text
def make_folds(train, n=5):
    # StratifiedGroupKFold on StudyInstanceUID, de-duped on report_md5 first
    # the gold 58 are never in a training fold
    # -> study_folds.csv
    ...

# ⚠️ *Need word Team ?*  <- the guard that keeps text out of inference
def assert_label_factory_only(module):
    # test.csv has no Report column
    # assert nothing here is importable by src/models/infer.py
    ...

### M4 — Encode once: frozen backbone → cached features
**goal** the one expensive pass, run ~3× for the whole project
**in** study tensors · **out** `features_<tag>.npy float16 [N,4,K,D]`

In [25]:
# src/models/encode.py

BACKBONES = {   # pre-attached Kaggle Models — scoring kernels have no internet
    "dinov2_s":    "metaresearch/dinov2/PyTorch/small/1   384-d, strongest frozen",
    "effnet_b3":   "timm/tf-efficientnet/PyTorch/b3/1     cheap 2.5D baseline",
    "radimagenet": "marwanmath/resnet-50-radimagenet      medical pretrain",
}

def to_model_input(vol_uint8):
    # 3 channels from ADJACENT slices (2.5D), not one slice repeated
    ...

def encode_all(studies, backbone="dinov2_s", tag="v1"):
    # per study/slot/slice: forward, keep [cls | mean_patch | max_patch], float16
    # encode the test set first and never subsample it
    # -> features_<tag>.npy [N,4,K,D], mask_<tag>.npy [N,4,K]
    ...

def cache_is_stale(tag):
    # hash (profile, backbone, K, size) into the tag
    # a stale cache looks like a bad idea, not a bad file
    ...

### M5 — The 12 specialist heads
**goal** each label gets its own model, sharing one encoder
**in** cached features + soft targets <br>⚠️ *Need word Team ?* · **out** OOF predictions, `submission.csv`

This is §5 in code form — read that section first for *why* each head differs.

In [26]:
# src/models/heads.py

def pool_features(feats, mask):
    # masked mean+max over the K slices of each slot
    # -> [N, 4*2*D]
    ...

def fit_heads(X, Y, W, folds):
    # ridge first: StandardScaler -> PCA(512) -> 12x weighted RidgeCV, ~20 s on a laptop
    # -> head_scaler.npz, head_pca.npy, head_ridge.npz   (never a pickle)
    ...

class SpecialistHead:   # -> nn.Module once torch lands (STRETCH, after Ridge plateaus)
    # attention over slices, then over slots, then linear -> 1 logit
    # that attention is the localisation signal M6 consumes
    ...

# ⚠️ *Need word Team ?*  <- y_soft and W both come from the reports
def train_head(label, feats, mask, y, w, folds):
    # loss = (BCE(pred, y_soft) * W[:, label]).sum() / W[:, label].sum()
    # per-label knobs: plane prior, slice window, profile, pos_weight
    ...

# ⚠️ *Need word Team ?*  <- OOF measures agreement with the reports, not with radiologists
def validate(oof, gold_58):
    # OOF measures agreement with the REPORTS -> regression guard, not a score estimate
    # the leaderboard is the only real validation; 5 submissions/day
    ...

def fuse_and_submit(oof, test_preds):
    # rank order only: no calibration, ensemble by rank mean
    # -> submission.csv, cols exactly ['StudyInstanceUID'] + TARGETS
    ...

### M6a — Weakly-supervised localisation (the colour)
**goal** a 3D heat volume per label, honestly gated
**in** features + head weights · **out** `data/_heat/<study>.npz`, `display_config.json`

In [27]:
# src/viz/localize.py

def contribution_map(patch_tokens, coef_label):
    # the head is linear, so logit = sum_p <token_p, coef> and the per-patch term IS the map
    # one einsum; no Grad-CAM, no backward pass, no GPU; keep the sign
    # -> signed (S, gh, gw), ~8 mm per token -> call it attention, never "the lesion"
    ...

def heat_volume(cam, alpha_slices, vol_shape):
    # weight each slice CAM by slice attention, upsample, smooth, mask to foreground
    ...

def qc_on_gold_58(heats, gold):
    # anatomy_lift, plane_agreement_mm, and a shuffled head that must score ~0
    # -> overlay_enabled per label; a label that fails ships with no colour
    ...

DISPLAY_CONTRACT = """
  overlay alpha is always multiplied by predicted confidence
  below tau -> no overlay at all, not a faint one
  the app returns 12 percentages + colour, never a sentence
"""

### M6b — The 2D / 3D viewer
**goal** one viewer, two render modes, over one precomputed pack
**in** volumes + heat · **out** `render_packs/<study>.npz`, Streamlit app

In [28]:
# src/viz/render_pack.py

def build_render_pack(study):
    # dumb renderer: one .npz, no DICOM, no model, no resampling
    # anatomy = ONE sagittal fluid-sensitive series (94.2% of studies), heat = its own CAM
    # no cross-plane fusion: three anisotropic anatomies resample into a blurry pancake
    # -> gray_iso (128,128,128), heat_iso (12,64,64,64), spacing, probs[12]
    ...

# src/viz/viewer.py — one viewer, 2D/3D toggle, same pack behind both
def view_2d(pack, label):
    # tri-planar + slider, heat overlaid. build this first, it debugs the pipeline visually
    ...

def view_3d(pack, label):
    # plotly marching-cubes blob + MPR planes, browser-side
    ...

def view_3d_demo(pack, label):   # STRETCH
    # PyVista/VTK precomputed to a GIF — interactive server-side VTK dies on stage
    ...

---
## 5. Process for each of the 12 models to recognize what is on an image

One shared encoder, twelve specialists. Each row is a different training recipe, not a
different backbone.

### Where the numbers in this table come from  <br>⚠️ *Need word Team ?*

No column is invented, but they do not share a source — that is why they don't look like
one file. Both label CSVs arrive via `pull_tier("T0_meta")`.

| column | source | computed as |
|---|---|---|
| **Prev.** | `data/meta/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv` | share of the 4,407 studies with `p > 0.5` |
| **Silence** | `data/meta/rsna-knee-llm-labels-pilkwang/report_labels_v2.csv` | share of the 4,406 rows with `__verdict == "UNK"` — i.e. the report never mentions the finding |
| **Supervision** | *derived, no source of its own* | `Silence` binned at **20 / 35 / 70 %** → 🟢 / 🟡 / 🟠 / 🔴 |
| **Primary slot** | `data/train_series.csv` | which `(Anatomical_Plane, Fluid_Sensitive)` series the study actually has |
| **published AUC** (`~0.69`, `~0.92`, …) | **outside — papers and public leaderboards** | **not reproducible from anything in `data/`**; treat as someone else's prior |

`Prev.` and `Silence` come from *different* datasets and are not interchangeable: computed
off pilkwang instead, `Prev.` for ACL reads 27.8 % rather than 20.8 %.

**§6 and §8 of `img_preprocess_all_models.ipynb` recompute every measured column and
raise if it stops matching this sheet.** The `Supervision` thresholds are pinned only to
gaps — see §8 there.

| # | Label | Primary slot | Anatomy to look at | Prev. | Silence | Supervision | Recipe |
|---|---|---|---|---|---|---|---|
| 1 | **ACL** | `SAG_FLUID` | intercondylar notch, mid-sagittal | 20.8% | 8.1% | 🟢 clean | narrow slice window on mid-sagittal; text is clean but the pixel signal is *hard* (published AUC ~0.69) |
| 2 | **MCL** | `COR_FLUID` | medial collateral ligament | 15.3% | 9.8% | 🟢 clean | same shape as ACL; also a weak pixel signal (~0.71) |
| 3 | **Medial Meniscus** | `SAG_FLUID` | posterior horn | 40.4% | **5.7%** | 🟢 best | best-supervised label in the set — make this the first head that works end-to-end |
| 4 | **Lateral Meniscus** | `SAG_FLUID`+`COR_FLUID` | lateral posterior horn | 15.7% | 10.0% | 🟢 clean | needs laterality fix or it learns nothing |
| 5 | **Medial OA** | `COR_FLUID` | medial tibiofemoral compartment | 37.2% | 25.5% | 🟡 ok | **best pixel signal in the competition** (~0.92). Cartilage loss + osteophytes are visually obvious |
| 6 | **Lateral OA** | `COR_FLUID` | lateral compartment | 27.3% | 33.1% | 🟡 ok | mirror of #5; share weights with it, differ only in the laterality channel |
| 7 | **PF OA** | **`AX_FLUID`** | patellofemoral joint | 45.7% | 18.5% | 🟢 clean | the one head that genuinely needs axial. Only 19.4% of studies have an axial non-fluid series |
| 8 | **Effusion** | `SAG_FLUID` | suprapatellar pouch | 59.8% | 9.9% | 🟢 clean | ~0.90 from pixels. Depends entirely on M2 not per-slice-windowing |
| 9 | **Synovitis** | `SAG_FLUID` | synovial lining | 12.6% | **84.2%** | 🔴 **none** | see below — this is the interesting one |
| 10 | **Baker's** | `SAG_FLUID`+`AX_FLUID` | **popliteal fossa (posterior)** | 24.7% | 45.9% | 🟠 poor | **the 130 mm crop cuts the cyst off.** Use the `context` full-FOV profile for this head only |
| 11 | **Contusion** | `SAG_FLUID` | bone marrow oedema | 17.5% | 20.8% | 🟡 ok | STIR/FS sequences only — on non-FS series the oedema is invisible |
| 12 | **Fracture** | `SAG_FLUID` | cortical break + oedema | **7.2%** | 56.4% | 🟠 poor | rarest label. Needs `pos_weight` or the head predicts all-negative and scores 0.5 |

### The three that decide our rank

Because the metric is an unweighted mean of 12 AUCs, **a label left at chance costs the same
~0.029 as any other**. Everyone optimises Effusion and Medial OA because they respond. The
rank is decided by the three nobody can train:

- **Synovitis (84% silent).** Almost no supervision exists. Do *not* train it on the report labels — they are noise, and fitting noise is worse than not fitting. Options, in order: <br>
  (a) predict from correlation with Effusion + Contusion, which co-occur with it;<br>
  (b) semi-supervised — pseudo-label from the model's own confident predictions;<br>
  (c) accept 0.5 and spend the time elsewhere. **Measure (a) against (c) on the gold 58 first.**<br>
- **Fracture (7.2% prevalent, 56% silent).** Rare *and* under-reported. `pos_weight`, heavy augmentation, and fluid-sensitive-FS series only.<br>
- **Baker's (46% silent).** Half the problem is ours, not the reports': our own crop removes the anatomy. Fixing the FOV for this one head is a cheap, real gain.<br>

**Recommended split, revisited:** we do not need 12 *training runs*. Heads 5+6 and 3+4 are
mirror pairs — train one model with a laterality channel and read both outputs. That is
**8 distinct recipes**, not 12.

---
## 6. Staged delivery ladder

Each stage ends with something that *works*. Never be in a state where nothing runs.

| Stage | Ships | Needs | Effort | Expected score |
|---|---|---|---|---|
| **0** | a **valid submission** from metadata only — per-label prevalence as a constant, or a logistic regression on `train_series.csv` counts. No images at all. **Plus the scoring-kernel skeleton**: a notebook that walks `/kaggle/input/test_series/`, does nothing useful, and writes a correctly-shaped `submission.csv` inside the time limit. | T0 (25 MB) | **½ day** | ~0.50–0.55 |
| **1** | First **image-based** submission. `SAG_FLUID` slot only, ~1,200-study subset, frozen DINOv2-S, Ridge heads. The scoring kernel decodes real DICOM. | M2 + one CPU build + one GPU encode | 4 days | **0.68–0.76** — ~90% of the final score arrives here |
| **2** | Full 4,407 × 4 slots, mean+max pooling, rank blend, LB-budgeted A/B of laterality on/off and slot count. | Kaggle GPU ×2 | 4 days | **0.74–0.80** |
| **3** | Fine-tune one shared encoder end-to-end; rank-mean ensemble across 2 backbones. | Kaggle GPU | 4–5 days | *uncertain — only attempt if Stage 2 plateaus* |
| **Demo** | M6: 2D/3D viewer + 12 percentages + precomputed orbit assets. Runs **parallel** from day 8; does not wait for Stage 2. | laptop | 5 days | graded, not scored |

**Calibrate expectations.** LB top ≈ **0.952**, 15th ≈ 0.943 — not reachable in a few weeks. A
fully-developed *public* pipeline reports macro **0.760**. A realistic target for us is
**0.74–0.80**. Gate each stage on *relative* movement — Stage 2 must beat Stage 1 by more than
the fold-to-fold spread — never on hitting an absolute number.

**Spend the 5 daily submissions deliberately**, because the LB is the only honest arbiter:
Mon Stage-0 baseline · Wed Stage-1 image arm · Thu laterality ON vs OFF (same seed, same folds)
· Fri 3-slot vs 4-slot. Log every one in `lb_log.csv` with its OOF in a **separate column** —
OOF measures agreement with an LLM, LB measures agreement with radiologists. Never quote them
on the same ladder.

**Do Stage 0 on day one.** A valid submission in the leaderboard removes all
end-of-project submission-format panic, and it is genuinely half a day of work.

**And time the scoring kernel from day one.** Every stage after this must answer "does it still
fit in 9 hours for 1,300 studies?". Measure `per_study_seconds` on 20 studies and multiply —
discovering at Stage 3 that inference takes 14 hours means throwing away Stage 3.

---
## 7. Who builds what

Contracts from §2 are the handoff points, so nobody blocks anybody.

| Person | Modules | First deliverable | Hands over |
|---|---|---|---|
| **A** | M1 + M2 | `manifest_series.parquet` + 58 gold volumes | `.npy` volumes + manifest |
| **B** ⚠️ *Need word Team ?* | M3 | `soft_targets.csv` + `label_audit.csv` | `y`, `w`, `folds` |
| **C** | M4 + M5 | cached features, then heads | OOF preds + `submission.csv` |
| **D** | M6a + M6b | 2D tri-planar viewer on gold 58 | the app |

**Unblocking trick:** D should not wait for C. Generate a **fake heat volume** (a Gaussian
blob at a plausible location) that satisfies the M6a contract, and build the entire viewer
against it. When the real CAMs arrive it is a one-line swap. Same for B → C: ship
`soft_targets.csv` full of random values on day 1 so C can build the training loop immediately.

---
## 8. Dependencies to add

Grouped by stage

```txt
# Stage 1 — preprocessing (laptop, Apple Silicon, py3.10.6)
pydicom>=3.0,<4
pylibjpeg>=2.0                # MANDATORY: JPEG-Lossless + JPEG-2000 series exist in this corpus
pylibjpeg-libjpeg>=2.1        # bare pydicom raises on .pixel_array without these
pylibjpeg-openjpeg>=2.4
pyarrow>=16,<21               # parquet manifests
opencv-python-headless        # cv2.resize INTER_AREA — ~10x faster than scipy.ndimage.zoom
python-dotenv                 # already used by kaggleFetch.ipynb
kaggle

# Stage 2 — models
torch>=2.5,<3                 # Shall I work with torch ? like the model imported are using pytorch
                              # but if we redo everything by ourselves ?
timm>=1.0
transformers>=4.44            # AutoModel for dinov2, with local_files_only=True on Kaggle

# Stage 3 / demo — visualisation  -> would be super cool if I manage to do it
streamlit>=1.40
plotly>=6.0                   # browser-side 3D, survives Streamlit Cloud
pyvista>=0.48                 # STRETCH: precomputed orbit assets only, NOT interactive server-side
```

---
## 9. Open questions — each with a default so nothing blocks

1. **Do we pull the 17.4 GB preprocessed mirror, or preprocess DICOM ourselves?**

2. **Frozen features or fine-tune?**
   Frozen ImageNet/DINOv2 features are **not** MSK-MRI features and will underperform
   fine-tuning by a wide margin. Frozen is scaffolding, not the destination.
3. **How many slices K per plane?** → *Default: 24.* Median series is 30 slices; 24 keeps the
   feature cache at ~600 MB and fits Kaggle's 20 GB `/kaggle/working`.
4. **Synovitis: attempt it or accept 0.5?** → *Default: measure the Effusion-correlation
   approach on the gold 58 (one afternoon), and only then decide.*
5. **Does anyone on the team have a CUDA machine?** → *Default: assume no.* All training is
   planned for Kaggle GPU with a 9-hour ceiling.
6. **Streamlit Cloud or local demo?** → *Default: local for demo day, Cloud as a bonus.*
   Precomputed assets mean the demo cannot fail live.

### Reminders that will save a week

- The scoring kernel has **no internet**. Every weight must be a pre-attached Kaggle Model.
- **9-hour** GPU limit, hidden test ≈1,300 studies → budget `per_study_seconds × 1300`.
- `test.csv` has **no Report column**. Reports are a training-time teacher only. ⚠️ *Need word Team ?*
- The metric reads **rank order only** — ensemble by rank, never by probability.
- Fix **laterality** before training, or 4 of 12 heads learn from an axis they cannot see.